<a href="https://colab.research.google.com/github/Lordyblade/BigData26_B_2411532010_IbrahimMousaDhani/blob/main/Praktikum5/BD_P05_2411532010_IbrahimMousaDhani.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Praktikum 5 — Pipeline Data Berita: dari Ingestion hingga MapReduce**


---
### **G. Persiapan Praktikum**

Pasang pustaka yang belum ada, lalu jalankan seluruh sel secara berurutan dari atas (banyak sel memakai variabel dari sel sebelumnya).

In [ ]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

---
### **H-2. Persiapan: Unduh Data dan Jalankan Portal Lokal**

Sel pertama mengunduh CSV dan melihat ukurannya. Sel kedua menyalakan portal berita mini di komputer kita (/berita?hal=N, 100 judul per halaman, serta robots.txt).

In [ ]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
            "dataset-judul-berita-indonesia/master/detik_news_title.csv")
if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas :", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom  :", df_sumber.shape)
print("Kolom         :", list(df_sumber.columns))

Ukuran berkas : 16.1 MB
Baris, kolom  : (91017, 4)
Kolom         : ['date', 'url', 'title', 'category']


In [ ]:
import http.server, urllib.parse, html

HALAMAN_UKURAN = 100  # judul per halaman

def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):  # matikan log
            pass

        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)

        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                        '</article>')
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                                 f'<nav data-total="{n_hal}"></nav></body></html>')
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal

server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


### **H-3. Crawling Portal Lokal: Latihan Aman**
Portal lokal selalu tersedia dan hasilnya sama untuk semua orang, sehingga cocok untuk berlatih mekanisme
crawling sebelum menyentuh situs sungguhan di H-4. Bila situs sungguhan nanti memblokir atau mengubah
susunannya, bagian ini tetap menjamin praktikum berjalan.

Langkah pertama crawler yang baik adalah membaca robots.txt. Sel berikut memeriksa dua alamat: halaman
berita biasa dan /arsip-privat/ yang dilarang oleh portal.

In [ ]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


Berikutnya, parse_halaman mengubah HTML menjadi data. Fungsi inilah To Do Anda: tentukan elemen
mana yang berisi judul, kategori, dan tanggal. Fungsi crawl_iter sudah siap pakai: ia memeriksa robots.txt,
mencoba ulang bila gagal, dan memberi jeda antar-halaman.

In [ ]:
def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append({
            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)

contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
    print(c)

Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


### **H-4. Crawling Berita Sungguhan dari Sitemap detik.com**
Sekarang kita mengambil data dari sumber sebenarnya. Kita tidak membuka halaman artikel satu per satu.
Detik menyediakan sitemap berita yang sudah memuat judul, tanggal terbit, dan kata kunci, sehingga cukup
beberapa berkas XML untuk mendapat puluhan judul. Berikut potongan satu entri dari sitemap
news.detik.com/berita/sitemap_news.xml pada uji dosen (29 Oktober 2026):

Langkah 1 — tulis parser (bagian kunci). Fungsi parse_sitemap_news mengubah satu berkas sitemap
menjadi daftar dict. Kategori diambil dari subdomain URL (news, inet, sport), sama seperti kolom kategori
pada CSV, dan id dari pola /d-<angka>/.

In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": m.group(1) if m else urlparse(url).netloc.split(".")[0],  # news, inet, sport, ...
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,  # 10 karakter pertama = YYYY-MM-DD
            "kata_kunci": kunci.get_text(strip=True) if kunci else "",
        })
    return hasil

Langkah 2 — uji tanpa internet. Sebelum menghubungi server, uji parser pada contoh XML kecil. Contoh
ini disusun dari tiga entri nyata hasil uji dosen; kata kunci hanya tersedia pada entri pertama.

In [ ]:
# Contoh XML: bentuk disalin dari sitemap berita asli (tiga entri hasil uji dosen, 29 Okt 2026).
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>
      <news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
      <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
      <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
      <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
      <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
      <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
    </news:news>
  </url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:",
      parse_sitemap_news(CONTOH_XML.encode("utf-8")) == parse_sitemap_news(CONTOH_XML))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi,mk,hukum,kepala daerah
Sama bila masukan berupa bytes: True


Langkah 3 — fungsi pengambilan yang sopan. Fungsi ini sudah siap pakai. Ada enam pengaman:
robots.txt dibaca per host, hanya tiga sitemap yang diambil (SITEMAP_DIPILIH), maksimum tujuh judul per
sitemap dan 20 judul total, jeda 2 detik antar-request, User-Agent yang menyebut identitas Anda, dan
berhenti seketika bila server membalas 403 atau 429. Ganti GANTI_DENGAN_EMAIL_ANDA dengan email
Anda.

In [ ]:
import urllib.robotparser

UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: ciburakatoprak@gmail.com)"
DELAY = 2.0              # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7     # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20         # batas seluruh crawling
SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]

_robots = {}
def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content  # bytes: pengkodean dibaca dari deklarasi XML

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

Langkah 4 — jalankan dan periksa. Hasil disimpan di folder contoh_crawl/, terpisah dari
lapisan_mentah/ agar angka acuan pada langkah berikutnya tidak berubah.

In [ ]:
try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    cek = pd.DataFrame(sampel)
    print("Judul terkumpul :", len(cek))
    print("Judul kosong    :", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat    :", cek["url"].duplicated().sum())
    print("Tanggal kosong  :", cek["tanggal"].isna().sum())
    print("Kategori        :", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))

Judul terkumpul : 20
Judul kosong    : 0
URL duplikat    : 6
Tanggal kosong  : 0
Kategori        : {'news': 7, '': 7, 'sport': 6}
                                                                        judul kategori    tanggal
      Muncul Usulan Komisioner Dipilih Presiden Lewat Pansel di HUT ke-73 KPU     news 2026-11-07
Paguyuban Brimob Puji Ketegasan Kapolri Tindak Listyo dkk di Kasus Penyadapan     news 2026-11-07
  Bamsoet: Munas FKPPI 2026 Akan Aklamasi Tetapkan Titiek Soeharto Jadi Ketum     news 2026-11-07


### **H-5. Channel ala Flume ke lapisan_mentah**
Untuk skala penuh, seluruh 911 halaman portal lokal di-crawl (bukan detik.com, yang tidak boleh dibebani 91
ribu judul sekaligus). Crawler adalah source, antrean queue.Queue adalah channel (maksimal 2.000 event),
dan fungsi sink menulis setiap 5.000 event menjadi satu berkas JSONL di lapisan_mentah/. Source dan sink
berjalan di thread terpisah. Bila channel penuh, source otomatis menunggu.

In [ ]:
RAW = Path("lapisan_mentah"); RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)  # menunggu bila channel penuh
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0
    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []
    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev); statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source); t_snk = threading.Thread(target=sink)
t_src.start(); t_snk.start(); t_src.join(); t_snk.join()
waktu = time.perf_counter() - t0

print("Event masuk channel :", f"{statistik['masuk']:,}")
print("Event ditulis sink  :", f"{statistik['keluar']:,}")
print("File batch          :", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3], "...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)
print("Total ukuran mentah :", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total   :", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel : 91,017
Event ditulis sink  : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 44.6 detik untuk 911 halaman


Event masuk dan event ditulis **harus sama (91.017)**. Dua angka yang berbeda berarti ada data yang hilang di tengah jalan.

### **H-6. Quality, Cleaning, dan Parquet Berpartisi**

Periksa dulu sebelum membersihkan. aturan_kualitas (**bagian kunci**) menghasilkan nilai benar/salah per baris untuk setiap aturan:
- judul 3-30 kata: jumlah kata antara 3 dan 30
- kategori dikenal: kategori ada di KATEGORI_DIKENAL
- url artikel valid: URL memuat pola /d-<angka> (URL # pada kategori inet gagal)

In [ ]:
mentah = pd.concat([pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))],
                   ignore_index=True)
print("Baris mentah:", f"{len(mentah):,}")

KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong"  : d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata"     : jml_kata.between(3, 30),
        "kategori dikenal"    : d["kategori"].isin(KATEGORI_DIKENAL),
        "tanggal valid"       : pd.to_datetime(d["tanggal"], format="%Y-%m-%d", errors="coerce").notna(),
        "url artikel valid"   : d["url"].fillna("").str.contains(r"/d-\d+", regex=True),
        "judul tidak duplikat": ~d["judul"].duplicated(keep=False),
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(f"{nama:<24} gagal: {(~lulus).sum():>6,} ({(~lulus).mean()*100:5.2f}%)")

Baris mentah: 91,017
judul tidak kosong       gagal:      0 ( 0.00%)
judul 3-30 kata          gagal:     78 ( 0.09%)
kategori dikenal         gagal:      0 ( 0.00%)
tanggal valid            gagal:      0 ( 0.00%)
url artikel valid        gagal:  5,640 ( 6.20%)
judul tidak duplikat     gagal:    235 ( 0.26%)


**Temuan kualitas data nyata:** aturan url artikel valid gagal untuk 5.640 baris, tepat sebanyak seluruh baris kategori inet (scraper menyimpan # sebagai URL). Judulnya masih berguna untuk analisis teks, jadi barisnya dipertahankan tetapi kolom url-nya dikosongkan agar tidak dipakai sebagai kunci.

**Pembersihan:** rapikan spasi, buang judul duplikat (simpan yang pertama), buang judul yang terlalu pendek/panjang, ubah tanggal ke datetime dengan format eksplisit, buat kolom bulan dan id.

In [ ]:
bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")  # kosong bila url rusak
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-\d+", regex=True))
print("Baris setelah dibersihkan:", f"{len(bersih):,}", "| dibuang:", f"{len(mentah) - len(bersih):,}")

Baris setelah dibersihkan: 90,818 | dibuang: 199


Terakhir, simpan sebagai Parquet berpartisi. Bagian kunci: pilih kolom partisi. Kode di bawah mencoba tiga
pilihan dan mencetak jumlah berkasnya, lalu menyimpan versi final dan mendemonstrasikan partition
pruning.

In [ ]:
import shutil, pyarrow.dataset as ds

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran

# BAGIAN KUNCI: bandingkan pilihan kolom partisi
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori = finance")

partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori = finance


Dasar memilih kategori: partisi ini menghasilkan 9 berkas dengan ukuran sangat beragam (2.433 sampai
32.321 judul), cukup sedikit untuk menghindari small file problem, dan nanti dipakai sebagai input split
MapReduce. Partisi kategori + bulan menghasilkan 54 berkas dengan total ukuran hampir sama, yaitu
banyak berkas kecil tanpa keuntungan. Partisi bulan hanya 6 berkas, tetapi pertanyaan kita menyangkut
kategori.

### **H-7. Query DuckDB dan Bentuk Dokumen**
Kolom kategori dibaca dari
nama folder (hive partitioning). Query kedua menghitung judul yang memuat kata "corona" per bulan.
Terakhir, satu baris diubah menjadi dokumen: bentuk fleksibel yang bisa menyimpan daftar kata di dalam
satu record.

In [ ]:
import duckdb
con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet', hive_partitioning=true)")

print(con.sql("SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC").df().to_string(index=False))
print()
print(con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan ORDER BY bulan""").df().to_string(index=False))

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


Bentuk tabel cocok untuk menghitung dan mengelompokkan. Bentuk dokumen cocok untuk menyimpan satu artikel beserta atribut turunannya (daftar kata, tag) tanpa mengubah skema tabel. Judul tentang corona memuncak pada Maret 2020; Juni hanya mencakup 13 hari pertama.

### **H-8. Mini MapReduce: Kasus Kantin**
Sebelum memproses 90 ribu judul, kerjakan kasus kantin dalam kode. Tiga fungsi (map_fn, shuffle, reduce_fn)
mewakili tiga tahap, dan setiap tahap dicetak agar terlihat bagaimana data berubah bentuk.

In [ ]:
kasir = [
    ["nasi goreng,1", "es teh,2", "mie ayam,1", "es teh,1"],      # kasir A
    ["nasi goreng,2", "es teh,1", "es teh,3", "mie ayam,1"],      # kasir B
    ["mie ayam,2", "nasi goreng,1", "es teh,1", "nasi goreng,1"], # kasir C
]

def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))

def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

# 1) MAP: setiap kasir memproses datanya sendiri
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE: kumpulkan nilai berdasarkan key
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE: gabungkan nilai per key
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# Combiner: jumlahkan dulu di setiap kasir sebelum dikirim
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())

setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner :", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner : 12
Pasangan dikirim dengan combiner: 9


Hasilnya sama dengan permainan kartu: 5, 8, dan 4. Pasangan yang dikirim turun dari 12 menjadi 9 dengan
combiner, tanpa mengubah hasil akhir.

### **H-9. Word Count Terdistribusi: Partisi sebagai Input Split**
Sekarang pola yang sama diterapkan pada judul berita. Satu input split adalah satu berkas partisi Parquet,
jadi ada 9 mapper. Inilah jembatan dengan Praktikum sebelumnya, cara kita mempartisi data menentukan
cara pekerjaan dibagi. Perhatikan bahwa ukuran split sangat tidak seimbang (news memiliki 32 ribu judul,
sport hanya 2 ribu).

In [ ]:
STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f"  {p.parent.name:<16} {n:>6,} judul")

def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)
hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")

print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f"  {k:<10} {n:>7,}")

top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f"  {k:<10} {n:>7,}")

# Cek silang dengan cara biasa (tanpa MapReduce)
cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
  kategori=finance 14,141 judul
  kategori=food     4,314 judul
  kategori=health   4,914 judul
  kategori=hot     16,290 judul
  kategori=inet     5,577 judul
  kategori=news    32,321 judul
  kategori=oto      4,375 judul
  kategori=sport    2,433 judul
  kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
  di          27,390
  corona      17,239
  ini         10,124
  yang         8,668
  dan          6,785
  ke           5,592
  tak          5,096
  dari         4,697
  jadi         4,486
  untuk        3,723

10 kata teratas (tanpa stopword):
  corona      17,239
  rp           3,154
  virus        3,075
  kasus        2,743
  polisi       2,732
  positif      2,682
  indonesia    2,504
  warga        2,495
  rumah        2,465
  jokowi       2,317

Sama dengan Counter biasa: True


### **H-10. Combiner dan Jumlah Data yang Berpindah**
Pertanyaan: apakah menjalankan mapper di beberapa proses membuat pekerjaan lebih cepat, dan apa
pengaruh combiner? Sel berikut mengukur waktu (median dari tiga kali jalan) dan menghitung pasangan
yang dikirim ke reducer. Angka pasangan stabil; waktu bervariasi antar-komputer dan antar-kali-jalan.

In [ ]:
from multiprocessing import Pool

def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0

print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")
for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses); waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}",
          f"{sum(len(x) for x in h):>20,}")
    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses); waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}",
          f"{sum(len(x) for x in h2):>20,}")

total = collections.Counter()
for c in h2:
    total.update(c)
print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          0.309              825,601
1 proses, dengan combiner         0.400               74,009
2 proses, tanpa combiner          0.921              825,601
2 proses, dengan combiner         0.426               74,009

Hasil dengan combiner sama persis: True


**Cara membaca hasil:** combiner memangkas data yang berpindah sekitar 11 kali lipat (825.601 → 74.009 pasangan) dengan hasil akhir persis sama. Pada contoh modul, 2 proses tanpa combiner bisa lebih lambat dari 1 proses karena biaya memindahkan 825 ribu pasangan antar-proses lebih besar daripada kerja mapper-nya; dengan combiner, 2 proses sedikit lebih cepat. Waktu berbeda di tiap komputer. Pelajaran utama: paralel tidak otomatis lebih cepat; yang mahal adalah memindahkan data.

### **H-11. Data Skew dan Beban Reducer**
Partitioner memutuskan reducer mana yang menerima sebuah key. Kita memakai hash (CRC32) sehingga
hasilnya sama di setiap komputer. Sel berikut menghitung beban setiap reducer, lalu membandingkan reducer
terberat dengan bagian yang adil bila jumlah reducer ditambah.

In [ ]:
R = 4  # jumlah reducer

def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b

def tampil(nama, b):
    tot = sum(b)
    print(f"{nama:<22}", "  ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))

tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f"  {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f"  {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")

tanpa combiner         R0=240,836 (29.2%)  R1=170,066 (20.6%)  R2=190,727 (23.1%)  R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%)  R1= 18,267 (24.7%)  R2= 18,530 (25.0%)  R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
    4 reducer: terberat 29.17% | bagian adil 25.00%
   16 reducer: terberat  8.82% | bagian adil  6.25%
   64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
  di        27,390 pasangan =  3.3% dari total, di reducer R0
  corona    17,239 pasangan =  2.1% dari total, di reducer R2
  ini       10,124 pasangan =  1.2% dari total, di reducer R0
  yang       8,668 pasangan =  1.0% dari total, di reducer R3
  dan        6,785 pasangan =  0.8% dari total, di reducer R3


**Cara membaca hasil:** dengan 4 reducer, reducer terberat menerima 29,2% pasangan (bagian adil 25%). Pada 64 reducer, reducer terberat 4,28% padahal bagian adilnya 1,56%. Penyebabnya: kata "di" sendirian sudah 3,3% dari seluruh pasangan dan satu key tidak bisa dipecah ke banyak reducer, menambah reducer tidak menyelesaikan skew dari satu key. Combiner membuat beban empat reducer hampir rata (24,7% sampai 25,6%), karena tiap kata hanya dikirim sekali per mapper.

### **H-12. Hadoop Streaming**
Hadoop Streaming menjalankan mapper dan reducer sebagai program biasa yang membaca stdin dan menulis
stdout. Sel pertama menulis berkas masukan (kategori dan judul dipisahkan tab) serta dua skrip mapper.py
dan reducer.py.

In [ ]:
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8")

Path("mapper.py").write_text('''import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')

print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))

Baris masukan: 90818


Uji mapper pada tiga baris pertama, lalu jalankan seluruh pipeline dengan pipe Linux. Perintah sort adalah
shuffle; LC_ALL=C memastikan urutan berdasarkan byte, sama seperti pengurutan key di Hadoop.

In [ ]:
%%bash
head -n 3 masukan.tsv | python3 mapper.py
time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
wc -l hasil_pipe.txt
sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5

kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785



real	0m1.170s
user	0m1.259s
sys	0m0.076s


Terakhir, membuktikan bahwa hasil pipe sama dengan hasil H-9:

In [ ]:
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)
print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Hasil pipe sama dengan hasil H-9: True


### **Opsional: menjalankan Hadoop asli (local mode)**

Jika waktu tersisa, jalankan mapper dan reducer yang sama pada Hadoop 3.3.6 sungguhan dalam local mode (satu mesin, tanpa HDFS dan YARN). Sel ini opsional dan memakan waktu unduh/instalasi.

In [ ]:
%%bash
sudo apt-get install -y openjdk-11-jdk-headless
wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
tar -xzf hadoop-3.3.6.tar.gz
export HADOOP_HOME=$PWD/hadoop-3.3.6
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64

rm -rf keluaran_hadoop
$HADOOP_HOME/bin/hadoop jar \
  $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input masukan.tsv -output keluaran_hadoop

ls keluaran_hadoop
diff <(LC_ALL=C sort hasil_pipe.txt) keluaran_hadoop/part-00000 && echo "SAMA"

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  openjdk-11-jre-headless
Suggested packages:
  openjdk-11-demo openjdk-11-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  openjdk-11-jdk-headless openjdk-11-jre-headless
0 upgraded, 2 newly installed, 0 to remove and 0 not upgraded.
Need to get 116 MB of archives.
After this operation, 258 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 openjdk-11-jre-headless amd64 11.0.32.1+1-1ubuntu1~24.04 [42.3 MB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 openjdk-11-jdk-headless amd64 11.0.32.1+1-1ubuntu1~24.04 [73.7 MB]
Fetched 116 MB in 1s (81.7 MB/s)
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 1

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
2026-11-07 09:00:35,707 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-11-07 09:00:36,025 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-11-07 09:00:36,025 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-11-07 09:00:36,062 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-11-07 09:00:36,409 INFO mapred.FileInputFormat: Total input files to process : 1
2026-11-07 09:00:36,434 INFO mapreduce.JobSubmitter: number of splits:1

### **H-13. Kata Teratas per Kategori, Ekspor ala Sqoop, dan Rangkuman**
Pada langkah terakhir, key diganti menjadi pasangan (kategori, kata), dengan stopword dibuang di sisi
mapper. Mapper memproses satu partisi, dan combiner dipasang di dalam mapper. Kemudian hasilnya
diekspor ke database SQLite, seperti sqoop export yang memindahkan hasil Hadoop ke database relasional.

In [ ]:
def mapper_kategori(path):
    """Key gabungan (kategori, kata); combiner dipasang di mapper."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(kategori, kata)] += 1
    return c

gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

per_kategori = collections.defaultdict(list)
for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))

for kat in sorted(per_kategori):
    top5 = sorted(per_kategori[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for n, k in top5))

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)


In [ ]:
import sqlite3

SIAP = Path("siap_pakai"); SIAP.mkdir(exist_ok=True)
baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

# "sqoop export": pindahkan hasil ke database relasional
db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)
print("Baris diekspor:", db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0])
print(db.execute("""SELECT kata, jumlah FROM kata_per_kategori
                    WHERE kategori = 'health' ORDER BY jumlah DESC LIMIT 3""").fetchall())
db.close()

Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


Hasil akhir tersimpan di folder `siap_pakai/`: berkas Parquet dan database `laporan.db`. Alur lengkap: portal → channel → dibersihkan → disimpan berpartisi → diproses MapReduce → dikirim ke database laporan.

---
### **M. Studi Kasus**

**Apa yang dibahas portal berita pada paruh pertama 2020?** Pimpinan redaksi meminta ringkasan satu halaman berdasarkan data judul, memakai temuan H-7 dan H-13.

Tugas: ringkasan 150–200 kata yang memuat (a) kapan topik ini memuncak, (b) kategori yang paling banyak membahasnya dan kategori yang bahasannya paling berbeda, (c) satu keterbatasan data.

Sel berikut mengumpulkan angka pendukung dari berita (DuckDB, H-7) dan per_kategori (H-13).

In [ ]:
# (a) Judul yang memuat "corona" per bulan (temuan H-7)
bulanan = con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan ORDER BY bulan""").df()
print(bulanan.to_string(index=False))
puncak = bulanan.loc[bulanan["judul_corona"].idxmax()]
print(f"\nPuncak: {puncak['bulan']} dengan {int(puncak['judul_corona']):,} judul")

# (b) Judul corona per kategori: jumlah dan proporsi terhadap seluruh judul kategori itu
tabel = con.sql("""
    SELECT kategori,
           COUNT(*) AS judul,
           SUM(CASE WHEN lower(judul) LIKE '%corona%' THEN 1 ELSE 0 END) AS judul_corona,
           ROUND(100.0 * SUM(CASE WHEN lower(judul) LIKE '%corona%' THEN 1 ELSE 0 END) / COUNT(*), 1) AS persen_corona
    FROM berita GROUP BY kategori ORDER BY judul_corona DESC""").df()
print("\n" + tabel.to_string(index=False))
total_c, total_j = tabel["judul_corona"].sum(), tabel["judul"].sum()
print(f"\nTotal judul corona: {int(total_c):,} dari {int(total_j):,} judul ({total_c/total_j*100:.1f}%)")

# (b) Temuan H-13: lima kata teratas (tanpa stopword) dan peringkat kata 'corona' per kategori
print("\nKata teratas per kategori (H-13):")
for kat in sorted(per_kategori):
    urut = sorted(per_kategori[kat], reverse=True)
    peringkat = next((i + 1 for i, (n, k) in enumerate(urut) if k == "corona"), None)
    print(f"  {kat:<8} peringkat 'corona': {peringkat} | teratas: " + ", ".join(k for n, k in urut[:5]))

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

Puncak: 2020-03 dengan 6,994 judul

kategori  judul  judul_corona  persen_corona
    news  32321        7532.0           23.3
  health   4914        2591.0           52.7
 finance  14141        2473.0           17.5
     hot  16290        1663.0           10.2
  travel   6453        1025.0           15.9
    inet   5577         847.0           15.2
     oto   4375         577.0           13.2
    food   4314         310.0            7.2
   sport   2433         228.0            9.4

Total judul corona: 17,246 dari 90,818 judul (19.0%)

Kata teratas per kategori (H-13):
  finance  peringkat 'corona': 1 | teratas: corona, rp, ri, harga, jokowi
  food     peringkat 'corona': 8 | teratas: makanan, makan, foto, galeri, enak
  health   peringkat 'corona': 1 | teratas: corona, virus, pasien, indonesia, meninggal
  hot      peringkat 'corona'

### Ringkasan untuk pimpinan redaksi

Pembahasan portal pada paruh pertama 2020 didominasi pandemi corona. Judul yang memuat kata "corona" melonjak dari 385 pada Januari menjadi 6.994 pada Maret (puncak), lalu turun menjadi 4.913 pada April dan 2.934 pada Mei. Juni baru mencakup 13 hari (943 judul), sehingga belum sebanding dengan bulan penuh. Secara keseluruhan terdapat 17.246 judul corona, sekitar 19% dari 90.818 judul bersih. Kategori news paling banyak membahasnya (7.532 judul), tetapi proporsi tertinggi ada di health: 52,7% judulnya memuat corona, dengan kata teratas virus dan pasien. Kategori yang bahasannya paling berbeda adalah food: hanya 7,2% judulnya membahas corona, dan lima kata teratasnya (makanan, makan, foto, galeri, enak) tidak memuat corona. Kategori oto juga berbeda, karena kata teratasnya mobil, bukan corona. Keterbatasan: data hanya berisi judul, bukan isi artikel, sehingga kata "corona" pada judul belum tentu berarti isinya membahas pandemi. Selain itu, Juni tidak penuh dan 5.577 judul inet tidak memiliki URL valid. Angka ini sebaiknya dibaca sebagai gambaran tren pemberitaan, bukan ukuran dampak pandemi.

---
## **N. Latihan**

### **Latihan 1 — Stopword domain**

Tambahkan 5 kata ke daftar stopword yang mengganggu hasil, lalu jalankan ulang H-13. Kata yang dipilih: `rp`, `ri`, `the` (muncul di antara kata teratas pada H-9/H-13), serta `foto` dan `galeri` (kata penanda format konten pada kategori food/travel, bukan topik berita). Daftar `STOPWORD` asli tidak diubah (dipakai latihan berikutnya); versi baru disimpan di `STOPWORD_LATIHAN`, dan berkas `siap_pakai/` tidak ditimpa.

In [ ]:
STOPWORD_BARU = {"rp", "ri", "the", "foto", "galeri"}
STOPWORD_LATIHAN = STOPWORD | STOPWORD_BARU

def mapper_kategori_sw(path, stopword):
    """Sama dengan mapper_kategori (H-13), tetapi daftar stopword menjadi parameter."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in stopword:
                c[(kategori, kata)] += 1
    return c

def top5_per_kategori(counter):
    pk = collections.defaultdict(list)
    for (kat, kata), n in counter.items():
        pk[kat].append((n, kata))
    return {kat: sorted(v, reverse=True)[:5] for kat, v in pk.items()}

gabung_baru = collections.Counter()
for p in splits:
    gabung_baru.update(mapper_kategori_sw(p, STOPWORD_LATIHAN))

top_lama = top5_per_kategori(gabung)       # hasil H-13
top_baru = top5_per_kategori(gabung_baru)  # hasil dengan stopword tambahan
print("Kata yang ditambahkan:", sorted(STOPWORD_BARU), "\n")
for kat in sorted(top_lama):
    lama = [k for n, k in top_lama[kat]]
    baru = [k for n, k in top_baru[kat]]
    print(f"{kat:<8} sebelum: " + ", ".join(f"{k} ({n:,})" for n, k in top_lama[kat]))
    print(f"{'':<8} sesudah: " + ", ".join(f"{k} ({n:,})" for n, k in top_baru[kat]))
    print(f"{'':<8} keluar : {sorted(set(lama) - set(baru)) or '-'} | masuk: {sorted(set(baru) - set(lama)) or '-'}\n")

Kata yang ditambahkan: ['foto', 'galeri', 'ri', 'rp', 'the'] 

finance  sebelum: corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
         sesudah: corona (2,470), harga (701), jokowi (642), mau (562), ekonomi (536)
         keluar : ['ri', 'rp'] | masuk: ['ekonomi', 'mau']

food     sebelum: makanan (699), makan (539), foto (458), galeri (438), enak (358)
         sesudah: makanan (699), makan (539), enak (358), bikin (343), rumah (311)
         keluar : ['foto', 'galeri'] | masuk: ['bikin', 'rumah']

health   sebelum: corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
         sesudah: corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
         keluar : - | masuk: -

hot      sebelum: corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
         sesudah: corona (1,658), anak (793), rumah (561), meninggal (545), film (489)
         keluar : ['the'] | masuk: ['film']

inet     sebelum: corona (845), v

 Kata yang masuk daftar stopword: `rp`, `ri`, `the`, `foto`, `galeri`. Lima kata teratas berubah pada 5 dari 9 kategori: finance (`ri`, `rp` keluar; `ekonomi`, `mau` masuk), food (`foto`, `galeri` keluar; `bikin`, `rumah` masuk), hot (`the` keluar; `film` masuk), oto (`rp` keluar; `honda` masuk), dan travel (`foto` keluar; `indonesia` masuk). Kategori health, inet, news, dan sport tidak berubah. Peringkat `corona` tidak berubah (tetap pertama, atau kedua pada oto), jadi kesimpulan utama sama; yang berubah hanya kata pendampingnya.

### **Latihan 2 — Per bulan**

Key diubah menjadi `(bulan, kata)`. Kolom `bulan` ada di setiap berkas partisi Parquet, jadi mapper tetap membaca satu partisi per mapper. Stopword memakai `STOPWORD` asli (seperti H-13).

In [ ]:
def mapper_bulan(path):
    """Key gabungan (bulan, kata); combiner dipasang di mapper."""
    d = pd.read_parquet(path, columns=["judul", "bulan"])
    c = collections.Counter()
    for t, b in zip(d["judul"], d["bulan"]):
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(b, kata)] += 1
    return c

gabung_bulan = collections.Counter()
for p in splits:
    gabung_bulan.update(mapper_bulan(p))

per_bulan = collections.defaultdict(list)
for (bln, kata), n in gabung_bulan.items():
    per_bulan[bln].append((n, kata))
urut_bulan = {b: sorted(v, reverse=True) for b, v in per_bulan.items()}

print("Lima kata teratas per bulan:")
for bln in sorted(urut_bulan):
    print(f"  {bln}", ", ".join(f"{k} ({n:,})" for n, k in urut_bulan[bln][:5]))

# Perbandingan Januari vs Mei (10 kata teratas)
jan = [k for n, k in urut_bulan["2020-01"][:10]]
mei = [k for n, k in urut_bulan["2020-05"][:10]]
print("\n10 teratas Januari:", jan)
print("10 teratas Mei    :", mei)
print("Hanya di Januari  :", [k for k in jan if k not in mei])
print("Hanya di Mei      :", [k for k in mei if k not in jan])

print(f"\n{'kata':<12}{'Jan':>8}{'Mei':>8}")
for k in dict.fromkeys(jan + mei):
    print(f"{k:<12}{gabung_bulan[('2020-01', k)]:>8,}{gabung_bulan[('2020-05', k)]:>8,}")

Lima kata teratas per bulan:
  2020-01 banjir (1,220), rp (616), polisi (554), jokowi (516), tahun (493)
  2020-02 corona (1,076), polisi (630), virus (596), rp (572), banjir (572)
  2020-03 corona (6,986), virus (1,340), positif (925), pasien (749), rumah (695)
  2020-04 corona (4,914), positif (828), pandemi (706), psbb (689), warga (609)
  2020-05 corona (2,935), kasus (640), positif (628), pandemi (596), warga (565)
  2020-06 corona (943), normal (471), new (465), kasus (315), rp (265)

10 teratas Januari: ['banjir', 'rp', 'polisi', 'jokowi', 'tahun', 'baru', 'mobil', 'minta', 'kpk', 'indonesia']
10 teratas Mei    : ['corona', 'kasus', 'positif', 'pandemi', 'warga', 'lebaran', 'rp', 'psbb', 'covid', 'polisi']
Hanya di Januari  : ['banjir', 'jokowi', 'tahun', 'baru', 'mobil', 'minta', 'kpk', 'indonesia']
Hanya di Mei      : ['corona', 'kasus', 'positif', 'pandemi', 'warga', 'lebaran', 'psbb', 'covid']

kata             Jan     Mei
banjir         1,220      59
rp               616   

Pada Januari, kata teratas adalah `banjir` (1.220), diikuti `rp`, `polisi`, `jokowi`, dan `tahun`. Pada Mei, kata teratas adalah `corona` (2.935), diikuti `kasus`, `positif`, `pandemi`, dan `warga`. Kata yang hanya masuk 10 teratas Januari: `banjir` (1.220 menjadi 59), `jokowi`, `tahun`, `baru`, `mobil`, `minta`, `kpk` (415 menjadi 97), dan `indonesia`. Kata yang hanya masuk 10 teratas Mei: `corona` (385 menjadi 2.935), `kasus`, `positif` (20 menjadi 628), `pandemi` (0 menjadi 596), `warga`, `lebaran` (9 menjadi 562), `psbb` (0 menjadi 522), dan `covid` (0 menjadi 522). Fokus pemberitaan bergeser dari banjir dan isu umum ke pandemi dan kebijakan pembatasannya.

### **Latihan 3 — Bigram**

Mapper menghasilkan pasangan kata berurutan dalam satu judul (misalnya `virus corona`) tanpa membuang stopword, sama seperti penghitungan 825.601 pasangan di H-9. Combiner = `Counter` lokal di tiap mapper (satu pasangan per bigram unik per partisi).

In [ ]:
def bigram(teks):
    k = tokenisasi(teks)
    return [f"{a} {b}" for a, b in zip(k, k[1:])]

def mapper_bigram(path):
    """Tanpa combiner: satu pasangan (bigram, 1) untuk setiap bigram."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(bg, 1) for t in judul for bg in bigram(t)]

def mapper_bigram_combiner(path):
    """Dengan combiner: jumlahkan dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(bigram(t))
    return c

hasil_bg = [mapper_bigram(p) for p in splits]
hasil_bg_c = [mapper_bigram_combiner(p) for p in splits]

pasangan_bg = sum(len(h) for h in hasil_bg)
pasangan_bg_c = sum(len(c) for c in hasil_bg_c)
print("Pasangan bigram tanpa combiner :", f"{pasangan_bg:,}")
print("Pasangan bigram dengan combiner:", f"{pasangan_bg_c:,}")
print(f"Penurunan                      : {pasangan_bg / pasangan_bg_c:.1f} kali lipat")

# Reduce: gabungkan hasil seluruh mapper
total_bg = collections.Counter()
for c in hasil_bg_c:
    total_bg.update(c)
print("Bigram unik                    :", f"{len(total_bg):,}")
print("'virus corona'                 :", f"{total_bg['virus corona']:,}")

print("\n10 bigram teratas:")
for bg, n in total_bg.most_common(10):
    print(f"  {bg:<20} {n:>6,}")

# Cek silang tanpa MapReduce
cek_bg = collections.Counter(bg for t in bersih["judul"] for bg in bigram(t))
print("\nSama dengan Counter biasa:", dict(cek_bg) == dict(total_bg))

Pasangan bigram tanpa combiner : 734,783
Pasangan bigram dengan combiner: 411,217
Penurunan                      : 1.8 kali lipat
Bigram unik                    : 349,114
'virus corona'                 : 2,915

10 bigram teratas:
  virus corona          2,915
  corona di             2,178
  positif corona        1,804
  di tengah             1,258
  pandemi corona        1,075
  di rumah              1,047
  new normal              917
  di indonesia            871
  rp juta                 700
  hari ini                627

Sama dengan Counter biasa: True


Tanpa combiner dihasilkan 734.783 pasangan bigram (tepat 825.601 − 90.818, karena judul dengan *n* kata menghasilkan *n* − 1 bigram). Dengan combiner turun menjadi 411.217 pasangan, hanya sekitar 1,8 kali lipat, jauh lebih kecil daripada 11 kali lipat pada kata tunggal di H-10. Penyebabnya: bigram jauh lebih beragam (349.114 bigram unik), sehingga sedikit yang berulang dalam satu partisi dan combiner kurang banyak menggabungkan. Bigram `virus corona` muncul 2.915 kali.

### **Latihan 4 — Jumlah reducer**

Jalankan H-11 dengan R = 2, 8, dan 32 (tanpa combiner, memakai `hasil_map` dari H-9), lalu buat tabel beban reducer terberat dibanding bagian yang adil.

In [ ]:
baris_tabel = []
for r in (2, 8, 32):
    b = beban(hasil_map, r)
    terberat = max(b) / sum(b) * 100
    adil = 100 / r
    baris_tabel.append({
        "jumlah reducer (R)": r,
        "terberat (%)": round(terberat, 2),
        "bagian adil (%)": round(adil, 2),
        "terberat / adil (kali)": round(terberat / adil, 2),
    })
tabel_reducer = pd.DataFrame(baris_tabel)
print(tabel_reducer.to_string(index=False))

 jumlah reducer (R)  terberat (%)  bagian adil (%)  terberat / adil (kali)
                  2         52.27            50.00                    1.05
                  8         16.83            12.50                    1.35
                 32          5.82             3.12                    1.86


**Catatan hasil.** Rasio reducer terberat terhadap bagian adil naik dari 1,05 kali (R = 2) ke 1,35 kali (R = 8) dan 1,86 kali (R = 32). Kata `di` saja sudah 3,3% dari seluruh pasangan dan tidak dapat dipecah, sehingga reducer yang memegangnya selalu menerima minimal 3,3%. Pada R = 32 bagian yang adil (3,12%) sudah lebih kecil daripada beban kata `di`, jadi menambah reducer tidak menyelesaikan skew yang berasal dari satu key.

### **Latihan 5 — Crawl ulang**

Tambahkan jeda 0,05 detik pada `crawl_iter` untuk 50 halaman pertama. Sel berikut mengukur waktunya (dan waktu tanpa jeda sebagai pembanding), lalu memperkirakan waktu 911 halaman dengan jeda 1 detik: `911 × (waktu kerja per halaman tanpa jeda + 1 detik)`.

In [ ]:
N_UJI = 50

t0 = time.perf_counter()
n_tanpa = len(list(crawl_iter(BASE, range(1, N_UJI + 1), delay=0.0)))
t_tanpa = time.perf_counter() - t0

t0 = time.perf_counter()
n_jeda = len(list(crawl_iter(BASE, range(1, N_UJI + 1), delay=0.05)))
t_jeda = time.perf_counter() - t0

print(f"{N_UJI} halaman, tanpa jeda   : {t_tanpa:6.2f} detik ({n_tanpa:,} judul)")
print(f"{N_UJI} halaman, jeda 0,05 dtk: {t_jeda:6.2f} detik ({n_jeda:,} judul)")
print(f"Selisih                      : {t_jeda - t_tanpa:6.2f} detik (jeda murni 50 x 0,05 = {N_UJI * 0.05:.2f} detik)")

kerja_per_halaman = t_tanpa / N_UJI
estimasi = N_HALAMAN * (kerja_per_halaman + 1.0)
print(f"\nKerja per halaman tanpa jeda : {kerja_per_halaman * 1000:.1f} ms")
print(f"Estimasi {N_HALAMAN} halaman, jeda 1 detik: {estimasi:,.0f} detik = {estimasi / 60:.1f} menit")
print(f"Batas bawah (hanya jeda)     : {N_HALAMAN * 1.0:,.0f} detik = {N_HALAMAN * 1.0 / 60:.1f} menit")

50 halaman, tanpa jeda   :   2.40 detik (5,000 judul)
50 halaman, jeda 0,05 dtk:   4.51 detik (5,000 judul)
Selisih                      :   2.11 detik (jeda murni 50 x 0,05 = 2.50 detik)

Kerja per halaman tanpa jeda : 48.0 ms
Estimasi 911 halaman, jeda 1 detik: 955 detik = 15.9 menit
Batas bawah (hanya jeda)     : 911 detik = 15.2 menit


### **Latihan 6 — Sitemap lain**

Alamat ketiga di `SITEMAP_DIPILIH` (sport) diganti dengan `sitemap_news.xml` dari bagian lain yang tercantum di `sitemap.xml`. Sel ini mencari alamat itu otomatis dari indeks sitemap (bagian dengan host berbeda dari news/inet/sport). Bila Anda ingin memilih sendiri, isi `SITEMAP_BARU` dengan satu alamat dari `sitemap.xml`. Aturan etika H-4 tetap berlaku (jeda 2 detik, maksimum 7 judul per sitemap, 20 judul total, berhenti pada 403/429); jangan dijalankan berulang dan jalankan bergiliran dengan teman sekelas.

> Alamat indeks di bawah adalah asumsi (`https://www.detik.com/sitemap.xml`); sesuaikan bila berbeda.

In [ ]:
INDEKS_SITEMAP = "https://www.detik.com/sitemap.xml"  # sesuaikan bila alamat indeks berbeda
SITEMAP_BARU = None  # isi manual dengan satu alamat sitemap_news.xml bila ingin memilih sendiri

def cari_sitemap_news_lain(indeks_url, sesi, host_dikecualikan):
    """Cari satu sitemap_news.xml pada indeks yang hostnya belum dipakai."""
    sup = BeautifulSoup(ambil_xml(indeks_url, sesi), "xml")
    for loc in sup.find_all("loc"):
        url = loc.get_text(strip=True)
        if url.endswith("sitemap_news.xml") and urlparse(url).netloc not in host_dikecualikan:
            return url
    return None

try:
    sesi6 = requests.Session()
    sesi6.headers["User-Agent"] = UA_DETIK
    if SITEMAP_BARU is None:
        host_dipakai = {urlparse(u).netloc for u in SITEMAP_DIPILIH}
        SITEMAP_BARU = cari_sitemap_news_lain(INDEKS_SITEMAP, sesi6, host_dipakai)
    if SITEMAP_BARU is None:
        raise RuntimeError("Tidak ada sitemap_news.xml dari bagian lain pada indeks")
    SITEMAP_LATIHAN6 = SITEMAP_DIPILIH[:2] + [SITEMAP_BARU]  # alamat ketiga (sport) diganti
    print("Sitemap yang dipakai:", *SITEMAP_LATIHAN6, sep="\n  ")

    sampel6 = []
    for sm in SITEMAP_LATIHAN6:
        if len(sampel6) >= BATAS_TOTAL:
            break
        sampel6 += parse_sitemap_news(ambil_xml(sm, sesi6))[:BATAS_PER_BAGIAN]
    sampel6 = sampel6[:BATAS_TOTAL]
except Exception as e:
    sampel6 = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel6:
    cek6 = pd.DataFrame(sampel6)
    print("\nJudul terkumpul :", len(cek6))
    print("Kategori yang muncul:", cek6["kategori"].value_counts().to_dict())
    print("\nAturan kualitas H-6 pada hasil crawl:")
    for nama, lulus in aturan_kualitas(cek6).items():
        print(f"  {nama:<24} gagal: {(~lulus).sum():>3} ({(~lulus).mean()*100:5.1f}%)")

Sitemap yang dipakai:
  https://news.detik.com/berita/sitemap_news.xml
  https://inet.detik.com/news/sitemap_news.xml
  https://finance.detik.com/berita-ekonomi-bisnis/sitemap_news.xml

Judul terkumpul : 20
Kategori yang muncul: {'news': 7, '': 7, 'finance': 6}

Aturan kualitas H-6 pada hasil crawl:
  judul tidak kosong       gagal:   0 (  0.0%)
  judul 3-30 kata          gagal:   0 (  0.0%)
  kategori dikenal         gagal:   7 ( 35.0%)
  tanggal valid            gagal:   0 (  0.0%)
  url artikel valid        gagal:   7 ( 35.0%)
  judul tidak duplikat     gagal:   0 (  0.0%)


### **Latihan 7 — Kata kunci**

Kolom `kata_kunci` berisi kata kunci tiap berita (dipisah koma). Mapper mengubah satu rekaman menjadi pasangan `(kata_kunci, 1)`, lalu shuffle dan reducer menjumlahkannya, pada hasil crawl H-4 (`contoh_crawl/detik_sitemap.jsonl`). Jika berkas itu tidak ada (crawl H-4 gagal atau belum dijalankan), sel memakai `CONTOH_XML` sebagai pengganti sementara.

In [ ]:
BERKAS_CRAWL = Path("contoh_crawl/detik_sitemap.jsonl")
if BERKAS_CRAWL.exists():
    rekaman = [json.loads(b) for b in open(BERKAS_CRAWL, encoding="utf-8")]
else:
    rekaman = parse_sitemap_news(CONTOH_XML)
    print("Berkas hasil crawl H-4 tidak ditemukan; memakai CONTOH_XML (3 entri) sebagai pengganti sementara.\n")

def mapper_kunci(rec):
    """Satu rekaman -> daftar pasangan (kata_kunci, 1)."""
    return [(k.strip().lower(), 1) for k in rec.get("kata_kunci", "").split(",") if k.strip()]

def reducer_kunci(kunci, daftar_nilai):
    return (kunci, sum(daftar_nilai))

# MAP
hasil_map_k = [mapper_kunci(r) for r in rekaman]
# SHUFFLE
shuffle_k = collections.defaultdict(list)
for h in hasil_map_k:
    for kunci, n in h:
        shuffle_k[kunci].append(n)
# REDUCE
hasil_k = dict(reducer_kunci(k, v) for k, v in shuffle_k.items())

print("Rekaman                    :", len(rekaman))
print("Rekaman punya kata kunci   :", sum(1 for h in hasil_map_k if h))
print("Pasangan dari mapper       :", sum(len(h) for h in hasil_map_k))
print("Kata kunci unik            :", len(hasil_k))
print("\nKata kunci terbanyak:")
for k, n in sorted(hasil_k.items(), key=lambda x: (-x[1], x[0]))[:10]:
    print(f"  {k:<30} {n:>3}")

Rekaman                    : 20
Rekaman punya kata kunci   : 20
Pasangan dari mapper       : 76
Kata kunci unik            : 64

Kata kunci terbanyak:
  ibl all indonesian               3
  ibl all indonesian 2026          3
  basket                           2
  ibl                              2
  jabodetabek                      2
  kasus penyadapan                 2
  kasus penyadapan prabowo         2
  lipsus smartfren                 2
  operator                         2
  telkomsel                        2


---
## **O. Tugas Mandiri — Inverted Index**

Bangun indeks terbalik dengan MapReduce: untuk setiap kata, simpan daftar id judul yang memuatnya. Mapper menghasilkan `(kata, id)`, reducer menggabungkan id menjadi daftar.

**Penanganan id kosong (kategori inet).** Kolom `id` kosong untuk seluruh 5.577 judul inet (URL-nya rusak). Keputusan: **baris inet tetap diindeks** dan tiap judul diberi **id sintetis** `10.000.000 + nomor baris pada partisi inet`.

Alasannya:
- Membuang baris inet menghilangkan satu kategori utuh dari indeks, padahal judulnya sah dan berguna untuk pencarian teks (alasan yang sama dengan keputusan mempertahankan baris di H-6).
- Id asli detik berkisar sekitar 4,8–5,1 juta, jadi id sintetis ≥ 10.000.001 tidak bertabrakan dengan id asli dan masih berupa bilangan bulat (daftar id tetap seragam dan terurut).
- Id sintetis bisa ditelusuri kembali ke judulnya lewat tabel pemetaan `peta_id_sintetis` yang disimpan di `siap_pakai/`.
- Penomoran per partisi aman karena hanya satu partisi (inet) yang memiliki id kosong; ini diperiksa dengan kode di bawah.

Mapper menghasilkan satu pasangan per kata unik dalam satu judul (kata yang muncul dua kali di judul yang sama tidak menggandakan id), dan semua kata dipertahankan (tanpa membuang stopword).

In [ ]:
import numpy as np

OFFSET_ID_SINTETIS = 10_000_000

# Pemeriksaan sebelum membangun indeks
print("Id kosong per kategori:", bersih.loc[bersih["id"].isna(), "kategori"].value_counts().to_dict())
print("Id asli terbesar      :", int(bersih["id"].max()), "(harus < offset", f"{OFFSET_ID_SINTETIS:,})")
assert bersih["id"].max() < OFFSET_ID_SINTETIS
print("Id asli duplikat      :", int(bersih["id"].dropna().duplicated().sum()))

splits_id_kosong = [p for p in splits if pd.read_parquet(p, columns=["id"])["id"].isna().any()]
print("Partisi dengan id kosong:", [p.parent.name for p in splits_id_kosong])
assert len(splits_id_kosong) == 1  # penomoran per partisi aman hanya jika satu partisi

def mapper_indeks(path):
    """Satu partisi -> pasangan (kata, id_dokumen). Id kosong diganti id sintetis."""
    d = pd.read_parquet(path, columns=["id", "judul"])
    pasangan = []
    for pos, (id_, judul) in enumerate(zip(d["id"], d["judul"])):
        doc_id = int(id_) if pd.notna(id_) else OFFSET_ID_SINTETIS + pos + 1
        for kata in set(tokenisasi(judul)):
            pasangan.append((kata, doc_id))
    return pasangan

def reducer_indeks(kata, daftar_id):
    """Gabungkan semua id untuk satu kata menjadi daftar terurut tanpa duplikat."""
    return (kata, sorted(set(daftar_id)))

# MAP
hasil_map_idx = [mapper_indeks(p) for p in splits]
print("\nPasangan (kata, id) dari semua mapper:", f"{sum(len(h) for h in hasil_map_idx):,}")
# SHUFFLE
shuffle_idx = collections.defaultdict(list)
for h in hasil_map_idx:
    for kata, doc_id in h:
        shuffle_idx[kata].append(doc_id)
# REDUCE
indeks = dict(reducer_indeks(k, v) for k, v in shuffle_idx.items())
print("Kata unik dalam indeks               :", f"{len(indeks):,}")

# Cek silang: panjang daftar id = jumlah judul yang memuat kata (dihitung tanpa MapReduce)
df_kata = collections.Counter(kata for t in bersih["judul"] for kata in set(tokenisasi(t)))
print("Panjang daftar id = jumlah judul pemuat kata:", {k: len(v) for k, v in indeks.items()} == dict(df_kata))
semua_id = {i for v in indeks.values() for i in v}
print("Judul terindeks:", f"{len(semua_id):,}", "dari", f"{len(bersih):,}", "judul bersih")

Id kosong per kategori: {'inet': 5577}
Id asli terbesar      : 5052369 (harus < offset 10,000,000)
Id asli duplikat      : 0
Partisi dengan id kosong: ['kategori=inet']

Pasangan (kata, id) dari semua mapper: 817,993
Kata unik dalam indeks               : 32,275
Panjang daftar id = jumlah judul pemuat kata: True
Judul terindeks: 90,818 dari 90,818 judul bersih


### **Hasil inverted index: tiga kata contoh beserta lima id pertamanya**

In [ ]:
judul_by_id = dict(zip(bersih.loc[bersih["id"].notna(), "id"].astype(int),
                       bersih.loc[bersih["id"].notna(), "judul"]))

for kata in ("corona", "virus", "jokowi"):
    daftar = indeks[kata]
    print(f"\n'{kata}' -> {len(daftar):,} judul | 5 id pertama: {daftar[:5]}")
    for i in daftar[:5]:
        print(f"    {i}  {judul_by_id[i]}")

# Bukti baris inet ikut terindeks lewat id sintetis
n_inet = sum(1 for i in indeks["corona"] if i > OFFSET_ID_SINTETIS)
print(f"\n'corona' pada judul inet (id sintetis > {OFFSET_ID_SINTETIS:,}): {n_inet:,} judul")


'corona' -> 17,228 judul | 5 id pertama: [4856495, 4856598, 4868317, 4868545, 4869163]
    4856495  WHO Sebut Penyakit Misterius di China Disebabkan oleh Virus Corona
    4856598  Seberapa Berbahaya Virus Corona di Balik 'Penyakit Misterius' di China?
    4868317  Peta Persebaran Virus Corona dari China, Ini Negara-negara yang Terdampak
    4868545  Virus Corona Baru Terkonfirmasi di AS, WHO Adakan Pertemuan Darurat
    4869163  Korut Bentengi Diri dari Virus Corona, Turis Dilarang Masuk!

'virus' -> 3,067 judul | 5 id pertama: [4856495, 4856598, 4860201, 4865102, 4865341]
    4856495  WHO Sebut Penyakit Misterius di China Disebabkan oleh Virus Corona
    4856598  Seberapa Berbahaya Virus Corona di Balik 'Penyakit Misterius' di China?
    4860201  Menkes Antisipasi Virus Baru di China: Bandara-Pelabuhan Siaga
    4865102  Virus 'Misterius' China Dilaporkan Menelan Korban Kedua
    4865341  Virus Baru China Diprediksi Sudah Menginfeksi Lebih dari 1.000 Manusia

'jokowi' -> 2,316 judul 

**Menyimpan indeks dan tabel pemetaan id sintetis ke lapisan `siap_pakai/`.**

In [ ]:
pd.DataFrame({
    "kata": list(indeks.keys()),
    "jumlah_judul": [len(v) for v in indeks.values()],
    "daftar_id": list(indeks.values()),
}).to_parquet(SIAP / "inverted_index.parquet", index=False)

d_inet = pd.read_parquet(splits_id_kosong[0], columns=["id", "judul"])
pos_kosong = np.flatnonzero(d_inet["id"].isna())
pd.DataFrame({
    "id_sintetis": OFFSET_ID_SINTETIS + pos_kosong + 1,
    "judul": d_inet["judul"].iloc[pos_kosong].values,
}).to_parquet(SIAP / "peta_id_sintetis.parquet", index=False)

print("Tersimpan di", SIAP, ":", sorted(p.name for p in SIAP.iterdir()))

Tersimpan di siap_pakai : ['inverted_index.parquet', 'kata_per_kategori.parquet', 'laporan.db', 'peta_id_sintetis.parquet']
